# Aula 14 — Breakpoints dinâmicos: pausar por REGRA

**Objetivo:** trocar a lista fixa (`interrupt_before`) por pausa condicional
dentro do nó — e conhecer a API nova (`interrupt` + `Command`).

**Pré-requisitos:** Aula 12 (pausa/retoma) e Aula 13 (`update_state`).

**Como usar:** rode em ordem. Autocontido, 100% offline (sem LLM, sem chave).
Comece pela CÉLULA 0.

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph
print('dependências prontas.')

## 1. Analogia: fiscal com régua, não com lista

Na Aula 12 o fiscal parava **todo** pedido. Agora ele tem régua: só para
conta **acima de R$ 50**. Conta baixa passa direto, sem ninguém perceber o fiscal.

| Fiscal | No LangGraph | O que é de verdade |
|---|---|---|
| régua de R$ 50 | **`raise NodeInterrupt(motivo)` no nó** | pausa condicional (API do curso, depreciada na v1.0) |
| interfone novo | **`interrupt(motivo)` + `Command(resume=...)`** | pausa com pergunta; a resposta volta como retorno |
| motivo no papel | **`get_state().tasks[].interrupts[].value`** | o 'porquê' da pausa, legível pelo humano |
| tentar passar sem carimbo | **retomar sem mudar** | reexecuta o nó → pausa DE NOVO (preso) |

Aviso honesto: o curso ensina `NodeInterrupt`; esta aula ensina os dois e
recomenda o novo (o velho sai na v2.0).

In [ ]:
# CÉLULA 1 — base: caixa com régua de R$ 50 (rode as duas contas)
import warnings
warnings.filterwarnings('ignore')  # cala o aviso de depreciação nas demos

from typing import TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.errors import NodeInterrupt
from langgraph.graph import START, END, StateGraph

LIMITE = 50.0

class Conta(TypedDict):
    entrada: str
    total: float

def caixa(state: Conta) -> dict:
    if state['total'] > LIMITE:
        raise NodeInterrupt(f'conta R$ {state["total"]} acima de R$ {LIMITE}')
    return {'total': state['total']}

b = StateGraph(Conta)
b.add_node('caixa', caixa)
b.add_edge(START, 'caixa'); b.add_edge('caixa', END)
fiscal_regua = b.compile(checkpointer=MemorySaver())

print('baixa:', fiscal_regua.invoke(
    {'entrada': 'pastel', 'total': 14.0}, {'configurable': {'thread_id': 'b1'}}))
print('(passou direto: a régua não mandou parar.)')

## 2. Conceito: a pausa mora no `if`, não na compilação

- **`raise NodeInterrupt(motivo)`**: o nó **não devolve** nada — ele *interrompe*.
  O `motivo` (string ou qualquer valor) fica guardado em
  `get_state().tasks[].interrupts[].value` para o humano ler.
- Nada de lista no `compile`: o grafo compila **limpo**; quem decide é o `if`
  com acesso ao estado (`state['total'] > LIMITE`).
- **Retomar sem mudar reexecuta o MESMO nó** → o `if` manda pausar de novo.
  Preso por desenho: a saída é `update_state` (Aula 13) ou resposta via `Command`.
- **API nova**: `decisao = interrupt('aprova?')` pausa **e pergunta**; na retomada
  com `Command(resume=True)`, a chamada devolve `True` — o humano responde
  *dentro* do fluxo, sem `update_state`.

In [ ]:
# CÉLULA 2 — conta alta pausa com motivo (rode e leia as tasks)
cfg = {'configurable': {'thread_id': 'b2'}}
for chunk in fiscal_regua.stream({'entrada': 'banquete', 'total': 55.0}, 
                                  cfg, stream_mode='updates'):
    print(list(chunk.keys()))
estado = fiscal_regua.get_state(cfg)
print('next:', estado.next)
for tarefa in estado.tasks:
    for pausa in tarefa.interrupts:
        print(f'motivo em {tarefa.name}: {pausa.value}')
assert estado.next == ('caixa',)
print()
print('Leitura: parou NO caixa, com o motivo guardado para o humano.')

## 3. Dissecando a pausa dinâmica

- `raise` (não `return`): interromper é exceção controlada, não update. O estado
  **não muda** na pausa — o checkpoint guarda o retrato de antes do nó.
- `tasks` (plural): cada tarefa pendente com seus `interrupts`. Em fan-out pode
  haver vários; aqui, um só (`caixa`).
- `pausa.value`: qualquer objeto Python (string, dict, dataclass). Use para
  mandar contexto ao painel de aprovação (`{'total': 55, 'mesa': 12}`).

In [ ]:
# CÉLULA 3 — retomar sem mudar trava; update destrava (rode as duas)
cfg3 = {'configurable': {'thread_id': 'b3'}}
list(fiscal_regua.stream({'entrada': 'banquete', 'total': 55.0}, cfg3))
list(fiscal_regua.stream(None, cfg3))   # retoma sem mudar nada
print('next após retomar sem mudar:', fiscal_regua.get_state(cfg3).next)
assert fiscal_regua.get_state(cfg3).next == ('caixa',)
print('preso: o if manda pausar de novo.')

fiscal_regua.update_state(cfg3, {'total': 40.0})   # fiscal concede desconto
print('após desconto:', fiscal_regua.invoke(None, cfg3))
print('OK — a saída do loop dinâmico é mudar o estado (Aula 13).')

## 4. Variações que quebram (todas executadas e verificadas)

### 4a. API nova: `interrupt()` pergunta, `Command` responde

Diferença real para o `NodeInterrupt`: o valor da pausa (`'aprova?'`) é a
**pergunta**, e a retomada traz a **resposta** como retorno da chamada.
Sem `update_state`: o humano responde *dentro* do fluxo.

In [ ]:
# CÉLULA 4 — API nova: pausa com pergunta, retoma com resposta
from langgraph.types import Command, interrupt

class ContaAprovada(TypedDict):
    entrada: str
    total: float
    aprovado: bool

def caixa_novo(state: ContaAprovada) -> dict:
    if state['total'] > LIMITE and not state.get('aprovado'):
        decisao = interrupt(f"conta R$ {state['total']} acima do limite: aprova?")
        if decisao is True:
            return {'aprovado': True}
    return {'total': state['total']}

bn = StateGraph(ContaAprovada)
bn.add_node('caixa', caixa_novo)
bn.add_edge(START, 'caixa'); bn.add_edge('caixa', END)
novo = bn.compile(checkpointer=MemorySaver())
cfgn = {'configurable': {'thread_id': 'nova'}}
list(novo.stream({'entrada': 'banquete', 'total': 55.0}, cfgn))
print('pausado. retomando com Command(resume=True)...')
list(novo.stream(Command(resume=True), cfgn))
print('final:', novo.get_state(cfgn).values)
assert novo.get_state(cfgn).values['aprovado'] is True
print('OK — a resposta do humano voltou como retorno do interrupt().')

### 4b. Estático vs dinâmico, lado a lado

- **Estático** (`interrupt_before=['caixa']`): pausa sempre, sem ler estado.
  Simples; burro.
- **Dinâmico** (`raise`/`interrupt` no nó): pausa por regra, com motivo.
  Flexível; exige tratar a retomada (update ou Command).
- Podem **combinar**: lista fixa para pontos críticos + regra para valores.

In [ ]:
# CÉLULA 5 — estático pausa até conta de R$ 1 (rode e compare)
be = StateGraph(Conta)
be.add_node('caixa', lambda s: {'total': s['total']})
be.add_edge(START, 'caixa'); be.add_edge('caixa', END)
fixo = be.compile(checkpointer=MemorySaver(), interrupt_before=['caixa'])
cfgf = {'configurable': {'thread_id': 'fixo'}}
list(fixo.stream({'entrada': 'pastel', 'total': 14.0}, cfgf))
print('next (conta baixa!):', fixo.get_state(cfgf).next)
assert fixo.get_state(cfgf).next == ('caixa',)
print('OK — o estático parou R$ 14 à toa. A régua (CÉLULA 1) deixaria passar.')

## 5. Erros clássicos

1. **Retomar sem mudar e achar que 'andou'.** Reexecuta o mesmo nó e pausa de
   novo (CÉLULA 3). Retomada dinâmica pede `update_state` ou `Command`.
2. **Usar `NodeInterrupt` em código novo.** Funciona, mas com aviso de
   depreciação (remoção na v2.0). Novo: `interrupt()` + `Command`.
3. **Retomar com input novo em vez de `Command`/`None`.** Input novo recomeça
   o fluxo; `Command(resume=...)` responde *dentro* da pausa.
4. **Motivo vazio ou técnico demais.** `pausa.value` é o que o humano lê no
   painel — mande contexto ('mesa 12, R$ 55'), não traceback.
5. **Regra que nunca libera.** `if` sem caminho de saída = thread presa para
   sempre. Toda regra de pausa precisa de regra de liberação.
6. **Pausar sem checkpointer.** Igual à Aula 12: pausa acontece, retomada morre.
   Dinâmico também exige sessão.

## 6. Exercício guiado — régua de R$ 100 para banquetes

Situação: mude o `LIMITE` para 100 e valide os 3 comportamentos: `14` passa
direto, `55` passa direto (abaixo da nova régua!) e `120` pausa com motivo.

In [ ]:
# CÉLULA 6 — exercício guiado (troque a régua)
LIMITE_EX = 100.0

def caixa_ex(state: Conta) -> dict:
    from langgraph.errors import NodeInterrupt as NI
    if state['total'] > LIMITE_EX:
        raise NI(f"conta R$ {state['total']} acima de R$ {LIMITE_EX}")
    return {'total': state['total']}

bex = StateGraph(Conta)
bex.add_node('caixa', caixa_ex)
bex.add_edge(START, 'caixa'); bex.add_edge('caixa', END)
gex = bex.compile(checkpointer=MemorySaver())

for total, tid in ((14.0, 'ex1'), (55.0, 'ex2'), (120.0, 'ex3')):
    cfgx = {'configurable': {'thread_id': tid}}
    list(gex.stream({'entrada': 'x', 'total': total}, cfgx))
    print(total, '-> next:', gex.get_state(cfgx).next)
assert gex.get_state({'configurable': {'thread_id': 'ex1'}}).next == ()
assert gex.get_state({'configurable': {'thread_id': 'ex2'}}).next == ()
assert gex.get_state({'configurable': {'thread_id': 'ex3'}}).next == ('caixa',)
print('OK — 14 e 55 passam, 120 pausa. A régua decide, não a lista.')

## 7. Desafio — fiscal com 2 réguas

Conta **acima de 100**: pausa pedindo aprovação (`interrupt`).
Conta **entre 50 e 100**: pausa só registrando motivo (sem pedir nada).
Abaixo de 50: passa direto.

- Dica: dois `if`s, dois motivos diferentes; o `interrupt` do meio pode levar
  um dict (`{'nivel': 'registro', 'total': ...}`) em vez de string.
- Valide os 3 totais (30, 70, 120) pelos `next` e pelos `tasks[].interrupts`.

## 8. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] Onde mora a pausa dinâmica: na compilação ou dentro do nó?
- [ ] O que `tasks[].interrupts[].value` carrega — e quem lê?
- [ ] Por que retomar sem mudar pausa de novo — e quais são as 2 saídas?
- [ ] `NodeInterrupt` vs `interrupt()` + `Command`: o que muda para o humano?
- [ ] Quando usar lista fixa, régua dinâmica, ou as duas?
- [ ] Na sua lanchonete, que réguas merecem pausa (valor? item sensível? horário)?

**Próxima aula (15 — Viagem no tempo):** navegar checkpoints, **reproduzir** o
passado e criar **forks** (ramos alternativos) — o debug definitivo do grafo.